In [1]:
%pip install -q unsloth datasets trl transformers==4.56.2 accelerate peft bitsandbytes pandas lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 93.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
from unsloth import FastLanguageModel
import torch

adapter_path = "/kaggle/input/datasets/ethanbell90/midterm-finetuned-model/final-model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_path,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
model.eval()

print("Model loaded and ready for batched inference")

==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded and ready for batched inference


In [4]:
import re
import xml.etree.ElementTree as ET
import pandas as pd
import time

SYSTEM_PROMPT = (
    "You are an SVG code generator. Given a description, output only valid SVG code, nothing else. "
    "Only use these elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, title, "
    "desc, style, pattern, marker, filter."
    "Keep the final SVG code strictly under 15000 characters."
    "Always use exactly these attributes in the opening tag: width=\"256\" height=\"256\" viewBox=\"0 0 200 200\""
)

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan", "title",
    "desc", "style", "pattern", "marker", "filter"
}

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def enforce_svg_attributes(svg):
    return re.sub(
        r'<svg[^>]*>',
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 200 200">',
        svg, count=1
    )

def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        
        # Check root tag, handling potential namespaces
        root_tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
        if root_tag != "svg":
            return False

        path_count = 0
        for elem in root.iter():
            # ElementTree prepends namespace like {http://www.w3.org/2000/svg} to tags
            tag_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
            
            # Condition 1: Tag must be in the allowed list
            if tag_name not in ALLOWED_TAGS:
                return False
                
            # Condition 2: Path count must not exceed 256
            if tag_name == "path":
                path_count += 1
                if path_count > 256:
                    return False
                    
        return True
    except ET.ParseError:
        return False

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="#cccccc"/>'
        '</svg>'
    )

def post_process(raw):
    m = re.search(r'<svg[\s\S]*?</svg>', raw, re.IGNORECASE)
    svg = m.group(0).strip() if m else ""
    if not svg and "<svg" in raw.lower():
        svg = raw[raw.lower().find("<svg"):]
        if not svg.rstrip().endswith("</svg>"):
            svg = svg + "</svg>"
    if svg:
        svg = enforce_svg_attributes(svg)
    if not is_valid_svg(svg) or len(svg) >= 16000:
        svg = fallback_svg()
    return svg

def generate_svg_batch(user_prompts, batch_size=16):
    print(f"Beginning inference with batch size of {batch_size}:")
    count = 1
    all_responses = []
    for i in range(0, len(user_prompts), batch_size):
        batch = user_prompts[i:i + batch_size]
        formatted_prompts = []
        for prompt in batch:
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt}
            ]
            formatted_string = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            formatted_prompts.append(formatted_string)

        inputs = tokenizer(
            formatted_prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=2048,
        ).to("cuda")

        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=2048,
            use_cache=True,
            temperature=0.6,
            repetition_penalty=1.10,
            top_p=0.9,
        )

        new_tokens = outputs[:, inputs.input_ids.shape[1]:]
        responses = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        all_responses.extend(responses)

        del inputs, outputs
        torch.cuda.empty_cache()
        print(f"Inference complete for {batch_size * count} samples.\n")
        count = count + 1
        
    return all_responses

In [ ]:
print(f"Beginning inference at time {time.time()}:")

test_df = pd.read_csv("/kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline/test.csv")
prompts = test_df["prompt"].tolist()

torch.cuda.empty_cache()
raw_output = generate_svg_batch(prompts, batch_size=50)
final_output = [post_process(r) for r in raw_output]

submission_df = pd.DataFrame({mn
    "id": test_df["id"],
    "svg": final_output
})

submission_df.to_csv("/kaggle/working/submission.csv", index=False)

print(f"Successfully generated {len(submission_df)} SVGs and saved to submission.csv!")

Beginning inference at time 1775066685.454584:
Beginning inference with batch size of 50:
Inference complete for 50 samples.

Inference complete for 100 samples.

Inference complete for 150 samples.

